# Nhóm 24: Ứng dụng Linked Open Data cho hệ thống ngân hàng Việt Nam
**Môn IT6390E Semantic Web, Đại học Bách khoa Hà Nội**

Notebook này là giao diện truy vấn (SPARQL terminal) cho bộ dữ liệu `vietnam-banking.ttl`. Notebook chạy được từ đầu đến cuối bằng *Run all*, trên Google Colab hoặc trên máy cá nhân.

| Phần | Nội dung |
|---|---|
| 0 | Cài đặt và nạp dữ liệu |
| 1 | Suy luận OWL RL và kiểm tra chất lượng dữ liệu |
| 2 | 16 truy vấn SPARQL: SELECT, FILTER, OPTIONAL, suy luận lớp, quan hệ ngược, COUNT/GROUP BY, ASK, CONSTRUCT, DESCRIBE, UNION, FILTER NOT EXISTS, SERVICE (federated) |
| 3 | Đóng gói kết quả |

## 0. Cài đặt và nạp dữ liệu

In [1]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "rdflib", "owlrl"], check=True)
print("Môi trường:", "Google Colab" if IN_COLAB else "máy cục bộ")

Môi trường: máy cục bộ


In [2]:
import os, zipfile
import pandas as pd
import owlrl
from rdflib import Graph
from rdflib.namespace import RDF, OWL

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)

DATA_FILE = "vietnam-banking.ttl"
candidates = [DATA_FILE, os.path.join("..", "data", DATA_FILE), os.path.join("data", DATA_FILE)]
data_path = next((p for p in candidates if os.path.exists(p)), None)

if data_path is None and IN_COLAB:
    from google.colab import files
    print("Hãy tải lên file vietnam-banking.ttl")
    uploaded = files.upload()
    data_path = next(iter(uploaded))

assert data_path, "Không tìm thấy vietnam-banking.ttl"

RESULTS_DIR = os.path.join("..", "results") if os.path.isdir(os.path.join("..", "results")) else "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Dữ liệu:", data_path)
print("Thư mục kết quả:", RESULTS_DIR)

Dữ liệu: ../data/vietnam-banking.ttl
Thư mục kết quả: ../results


## 1. Suy luận OWL RL và kiểm tra chất lượng dữ liệu
- `g_raw`: đồ thị gốc, chỉ gồm các bộ ba đã khai báo. Dùng cho thống kê, liên kết, CONSTRUCT, DESCRIBE.
- `g`: đồ thị sau suy luận OWL RL. Dùng cho các truy vấn cần lớp được định nghĩa (`DigitalBank`, `StateOwnedBank`, `ForeignBank`) và quan hệ ngược (`owl:inverseOf`).

In [3]:
g_raw = Graph()
g_raw.parse(data_path, format="turtle")

g = Graph()
for triple in g_raw:
    g.add(triple)
owlrl.DeductiveClosure(owlrl.OWLRL_Semantics).expand(g)

print("Số bộ ba trước suy luận:", len(g_raw))
print("Số bộ ba sau suy luận  :", len(g))
print("Số bộ ba được suy luận  :", len(g) - len(g_raw))
print("Cá thể thuộc owl:Nothing (mâu thuẫn):", len(set(g.subjects(RDF.type, OWL.Nothing))))

Số bộ ba trước suy luận: 1067
Số bộ ba sau suy luận  : 4570
Số bộ ba được suy luận  : 3503
Cá thể thuộc owl:Nothing (mâu thuẫn): 0


In [4]:
PREFIXES = """
PREFIX :        <https://data.example.org/vietnam-banking/>
PREFIX bank:    <https://data.example.org/vietnam-banking/bank/>
PREFIX loc:     <https://data.example.org/vietnam-banking/location/>
PREFIX bt:      <https://data.example.org/vietnam-banking/bank-type/>
PREFIX sv:      <https://data.example.org/vietnam-banking/service/>
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX xsd:     <http://www.w3.org/2001/XMLSchema#>
PREFIX schema:  <https://schema.org/>
PREFIX void:    <http://rdfs.org/ns/void#>
PREFIX dcterms: <http://purl.org/dc/terms/>
PREFIX wdt:     <http://www.wikidata.org/prop/direct/>
"""

def sparql(query, graph=None):
    """SPARQL terminal: chạy một truy vấn SELECT bất kỳ, trả về DataFrame."""
    graph = g if graph is None else graph
    result = graph.query(PREFIXES + query)
    return pd.DataFrame([[("" if v is None else str(v)) for v in row] for row in result],
                        columns=[str(v) for v in result.vars])

def run_query(title, query, csv_name, graph=None):
    df = sparql(query, graph)
    print(title)
    print("Số kết quả:", len(df))
    display(df)
    df.to_csv(os.path.join(RESULTS_DIR, csv_name), index=False, encoding="utf-8-sig")
    return df

In [5]:
# Kiểm tra ràng buộc của ontology trên dữ liệu gốc
checks = {
    "Ngân hàng": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank a :Bank . FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/")) }''',
    "Thiếu loại ngân hàng": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank a :Bank . FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/")) FILTER NOT EXISTS { ?bank :hasBankType ?t } }''',
    "Thiếu trụ sở": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank a :Bank . FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/")) FILTER NOT EXISTS { ?bank :hasHeadquarters ?h } }''',
    "Nhiều hơn 1 loại": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank :hasBankType ?a, ?b . FILTER(?a != ?b) }''',
    "Nhiều hơn 1 trụ sở": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank :hasHeadquarters ?a, ?b . FILTER(?a != ?b) }''',
    "Thiếu nhãn rdfs:label": '''SELECT (COUNT(DISTINCT ?bank) AS ?n) WHERE { ?bank a :Bank . FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/")) FILTER NOT EXISTS { ?bank rdfs:label ?l } }''',
    "Địa điểm không được dùng": '''SELECT (COUNT(DISTINCT ?l) AS ?n) WHERE { ?l a :Location . FILTER NOT EXISTS { ?x :hasHeadquarters|:operatesInCountry ?l } }''',
}
quality = pd.DataFrame([(k, int(sparql(q, g_raw)["n"][0])) for k, q in checks.items()], columns=["Kiểm tra", "Số lượng"])
display(quality)

,Kiểm tra,Số lượng
0,Ngân hàng,30
1,Thiếu loại ngân hàng,0
2,Thiếu trụ sở,0
3,Nhiều hơn 1 loại,0
4,Nhiều hơn 1 trụ sở,0
5,Thiếu nhãn rdfs:label,0
6,Địa điểm không được dùng,0


## 2. Truy vấn SPARQL

### Truy vấn 1: Danh sách ngân hàng
SELECT cơ bản với OPTIONAL và ORDER BY.

In [6]:
df1 = run_query(
    "TRUY VẤN 1: DANH SÁCH NGÂN HÀNG",
    """
    SELECT ?id ?nameVi ?nameEn ?typeEn
    WHERE {
        ?bank a :Bank ;
              :bankId ?id ;
              :nameVi ?nameVi .
        OPTIONAL { ?bank :nameEn ?nameEn }
        OPTIONAL { ?bank :hasBankType/rdfs:label ?typeEn . FILTER(LANG(?typeEn) = "en") }
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_01_all_banks.csv",
    graph=g_raw,
)

TRUY VẤN 1: DANH SÁCH NGÂN HÀNG
Số kết quả: 30


,id,nameVi,nameEn,typeEn
0,VBSP,Ngân hàng Chính sách Xã hội Việt Nam,Vietnam Bank for Social Policies,Policy bank
1,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,Vietnam Bank for Agriculture and Rural Development,State-owned or state-controlled commercial bank
2,VDB,Ngân hàng Phát triển Việt Nam,Vietnam Development Bank,Policy bank
3,ABBANK,Ngân hàng TMCP An Bình,An Binh Commercial Joint Stock Bank,Joint-stock commercial bank
4,BACABANK,Ngân hàng TMCP Bắc Á,Bac A Commercial Joint Stock Bank,Joint-stock commercial bank
5,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,Vietnam Joint Stock Commercial Bank for Industry and Trade,State-owned or state-controlled commercial bank
6,MSB,Ngân hàng TMCP Hàng Hải Việt Nam,Vietnam Maritime Commercial Joint Stock Bank,Joint-stock commercial bank
7,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam,Vietnam Technological and Commercial Joint Stock Bank,Joint-stock commercial bank
8,LPBANK,Ngân hàng TMCP Lộc Phát Việt Nam,Fortune Vietnam Joint Stock Commercial Bank,Joint-stock commercial bank
9,NAMABANK,Ngân hàng TMCP Nam Á,Nam A Commercial Joint Stock Bank,Joint-stock commercial bank


### Truy vấn 2: Ngân hàng thương mại nhà nước
Dùng lớp `:StateOwnedBank`. Lớp này không được gán trực tiếp mà do reasoner suy ra từ định nghĩa `owl:equivalentClass`.

In [7]:
df2 = run_query(
    "TRUY VẤN 2: NGÂN HÀNG THƯƠNG MẠI NHÀ NƯỚC",
    """
    SELECT ?id ?nameVi ?nameEn
    WHERE {
        ?bank a :StateOwnedBank ;
              :bankId ?id ;
              :nameVi ?nameVi .
        OPTIONAL { ?bank :nameEn ?nameEn }
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_02_state_owned_banks.csv",
    graph=g,
)

TRUY VẤN 2: NGÂN HÀNG THƯƠNG MẠI NHÀ NƯỚC
Số kết quả: 5


,id,nameVi,nameEn
0,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,Vietnam Bank for Agriculture and Rural Development
1,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,Vietnam Joint Stock Commercial Bank for Industry and Trade
2,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam,Joint Stock Commercial Bank for Foreign Trade of Vietnam
3,BIDV,Ngân hàng TMCP Đầu tư và Phát triển Việt Nam,Joint Stock Commercial Bank for Investment and Development of Vietnam
4,VCBNEO,Ngân hàng TNHH MTV Ngoại thương Công nghệ số,Vietcombank Neo


### Truy vấn 3: Ngân hàng có trụ sở tại Hà Nội
So khớp trực tiếp với IRI nội bộ `loc:hanoi`. Cách này tránh nhân bản kết quả do `owl:sameAs` sinh ra (Hà Nội có 3 IRI tương đương sau suy luận).

In [8]:
df3 = run_query(
    "TRUY VẤN 3: NGÂN HÀNG CÓ TRỤ SỞ TẠI HÀ NỘI",
    """
    SELECT ?id ?nameVi
    WHERE {
        ?bank :hasHeadquarters loc:hanoi ;
              :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_03_hanoi_headquarters.csv",
    graph=g,
)

TRUY VẤN 3: NGÂN HÀNG CÓ TRỤ SỞ TẠI HÀ NỘI
Số kết quả: 20


,id,nameVi
0,VBSP,Ngân hàng Chính sách Xã hội Việt Nam
1,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam
2,VDB,Ngân hàng Phát triển Việt Nam
3,ABBANK,Ngân hàng TMCP An Bình
4,BACABANK,Ngân hàng TMCP Bắc Á
5,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam
6,MSB,Ngân hàng TMCP Hàng Hải Việt Nam
7,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam
8,LPBANK,Ngân hàng TMCP Lộc Phát Việt Nam
9,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam


### Truy vấn 4: Ngân hàng cung cấp ngân hàng số
Dùng lớp `:DigitalBank` do reasoner suy ra (ngân hàng có `providesService sv:digital-banking`).

In [9]:
df4 = run_query(
    "TRUY VẤN 4: NGÂN HÀNG CUNG CẤP NGÂN HÀNG SỐ",
    """
    SELECT ?id ?nameVi
    WHERE {
        ?bank a :DigitalBank ;
              :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_04_digital_banks.csv",
    graph=g,
)

TRUY VẤN 4: NGÂN HÀNG CUNG CẤP NGÂN HÀNG SỐ
Số kết quả: 28


,id,nameVi
0,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam
1,ABBANK,Ngân hàng TMCP An Bình
2,BACABANK,Ngân hàng TMCP Bắc Á
3,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam
4,MSB,Ngân hàng TMCP Hàng Hải Việt Nam
5,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam
6,LPBANK,Ngân hàng TMCP Lộc Phát Việt Nam
7,NAMABANK,Ngân hàng TMCP Nam Á
8,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam
9,HDBANK,Ngân hàng TMCP Phát triển Thành phố Hồ Chí Minh


### Truy vấn 5: Ngân hàng thành lập trước năm 1991
Lớp `:EstablishedBefore1991Bank` dùng ràng buộc kiểu dữ liệu (`xsd:maxInclusive`). OWL RL không hỗ trợ loại ràng buộc này, nên lớp chỉ được suy ra bởi reasoner OWL 2 DL như HermiT trong Protégé. Ở đây dùng FILTER với cùng điều kiện.

In [10]:
df5 = run_query(
    "TRUY VẤN 5: NGÂN HÀNG THÀNH LẬP TRƯỚC NĂM 1991",
    """
    SELECT ?id ?nameVi ?foundedYear
    WHERE {
        ?bank a :Bank ;
              :bankId ?id ;
              :nameVi ?nameVi ;
              :foundedYear ?foundedYear .
        FILTER(?foundedYear <= 1990)
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?foundedYear ?nameVi
    """,
    "query_05_established_before_1991.csv",
    graph=g_raw,
)

TRUY VẤN 5: NGÂN HÀNG THÀNH LẬP TRƯỚC NĂM 1991
Số kết quả: 6


,id,nameVi,foundedYear
0,BIDV,Ngân hàng TMCP Đầu tư và Phát triển Việt Nam,1957
1,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam,1963
2,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,1988
3,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,1988
4,EXIMBANK,Ngân hàng TMCP Xuất Nhập khẩu Việt Nam,1989
5,HDBANK,Ngân hàng TMCP Phát triển Thành phố Hồ Chí Minh,1990


### Truy vấn 6: Liên kết tới Wikidata và DBpedia
Liệt kê các liên kết `owl:sameAs` đã khai báo (5★).

In [11]:
df6 = run_query(
    "TRUY VẤN 6: LIÊN KẾT TỚI WIKIDATA VÀ DBPEDIA",
    """
    SELECT ?id ?nameVi ?dataset ?externalResource
    WHERE {
        ?bank a :Bank ;
              :bankId ?id ;
              :nameVi ?nameVi ;
              owl:sameAs ?externalResource .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
        BIND(IF(STRSTARTS(STR(?externalResource), "http://www.wikidata.org/"), "Wikidata", "DBpedia") AS ?dataset)
    }
    ORDER BY ?nameVi ?dataset
    """,
    "query_06_external_links.csv",
    graph=g_raw,
)

TRUY VẤN 6: LIÊN KẾT TỚI WIKIDATA VÀ DBPEDIA
Số kết quả: 32


,id,nameVi,dataset,externalResource
0,VBSP,Ngân hàng Chính sách Xã hội Việt Nam,Wikidata,http://www.wikidata.org/entity/Q10800262
1,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,DBpedia,http://dbpedia.org/resource/Agribank_(Vietnam)
2,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,Wikidata,http://www.wikidata.org/entity/Q1924723
3,VDB,Ngân hàng Phát triển Việt Nam,Wikidata,http://www.wikidata.org/entity/Q10800267
4,ABBANK,Ngân hàng TMCP An Bình,Wikidata,http://www.wikidata.org/entity/Q10800288
5,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,DBpedia,http://dbpedia.org/resource/Vietinbank
6,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,Wikidata,http://www.wikidata.org/entity/Q7928459
7,MSB,Ngân hàng TMCP Hàng Hải Việt Nam,Wikidata,http://www.wikidata.org/entity/Q10800277
8,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam,DBpedia,http://dbpedia.org/resource/Techcombank
9,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam,Wikidata,http://www.wikidata.org/entity/Q10541776


### Truy vấn 7: Quan hệ ngược dịch vụ → ngân hàng
`:isProvidedBy` không có trong dữ liệu gốc. Reasoner suy ra từ `owl:inverseOf :providesService`.

In [12]:
df7 = run_query(
    "TRUY VẤN 7: QUAN HỆ NGƯỢC DỊCH VỤ → NGÂN HÀNG",
    """
    SELECT ?id ?nameVi
    WHERE {
        sv:digital-banking :isProvidedBy ?bank .
        ?bank :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_07_inverse_service.csv",
    graph=g,
)

TRUY VẤN 7: QUAN HỆ NGƯỢC DỊCH VỤ → NGÂN HÀNG
Số kết quả: 28


,id,nameVi
0,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam
1,ABBANK,Ngân hàng TMCP An Bình
2,BACABANK,Ngân hàng TMCP Bắc Á
3,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam
4,MSB,Ngân hàng TMCP Hàng Hải Việt Nam
5,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam
6,LPBANK,Ngân hàng TMCP Lộc Phát Việt Nam
7,NAMABANK,Ngân hàng TMCP Nam Á
8,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam
9,HDBANK,Ngân hàng TMCP Phát triển Thành phố Hồ Chí Minh


### Truy vấn 8: Quan hệ ngược trụ sở → ngân hàng
`:isHeadquartersOf` được suy ra từ `owl:inverseOf :hasHeadquarters`. Kết quả phải trùng với truy vấn 3.

In [13]:
df8 = run_query(
    "TRUY VẤN 8: QUAN HỆ NGƯỢC TRỤ SỞ → NGÂN HÀNG",
    """
    SELECT ?id ?nameVi
    WHERE {
        loc:hanoi :isHeadquartersOf ?bank .
        ?bank :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?nameVi
    """,
    "query_08_inverse_headquarters.csv",
    graph=g,
)

TRUY VẤN 8: QUAN HỆ NGƯỢC TRỤ SỞ → NGÂN HÀNG
Số kết quả: 20


,id,nameVi
0,VBSP,Ngân hàng Chính sách Xã hội Việt Nam
1,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam
2,VDB,Ngân hàng Phát triển Việt Nam
3,ABBANK,Ngân hàng TMCP An Bình
4,BACABANK,Ngân hàng TMCP Bắc Á
5,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam
6,MSB,Ngân hàng TMCP Hàng Hải Việt Nam
7,TECHCOMBANK,Ngân hàng TMCP Kỹ thương Việt Nam
8,LPBANK,Ngân hàng TMCP Lộc Phát Việt Nam
9,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam


### Truy vấn 9: Số ngân hàng theo loại
COUNT và GROUP BY. Tổng phải bằng 30.

In [14]:
df9 = run_query(
    "TRUY VẤN 9: SỐ NGÂN HÀNG THEO LOẠI",
    """
    SELECT ?bankType (COUNT(DISTINCT ?bank) AS ?numberOfBanks)
    WHERE {
        ?bank a :Bank ;
              :hasBankType/rdfs:label ?bankType .
        FILTER(LANG(?bankType) = "vi")
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    GROUP BY ?bankType
    ORDER BY DESC(?numberOfBanks)
    """,
    "query_09_count_banks_by_type.csv",
    graph=g_raw,
)

TRUY VẤN 9: SỐ NGÂN HÀNG THEO LOẠI
Số kết quả: 5


,bankType,numberOfBanks
0,Ngân hàng thương mại cổ phần,20
1,Ngân hàng thương mại nhà nước/chi phối,5
2,Ngân hàng thương mại trong nước thuộc sở hữu tư nhân,2
3,Ngân hàng chính sách,2
4,Ngân hàng 100% vốn nước ngoài,1


In [15]:
print("Tổng số ngân hàng:", df9["numberOfBanks"].astype(int).sum())

Tổng số ngân hàng: 30


### Truy vấn 10: ASK kiểm tra dataset có liên kết 5★
ASK trả về đúng hoặc sai.

In [16]:
ask_query = """
ASK {
    ?bank a :Bank ;
          owl:sameAs ?external .
    FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    FILTER(STRSTARTS(STR(?external), "http://www.wikidata.org/entity/")
        || STRSTARTS(STR(?external), "http://dbpedia.org/resource/"))
}
"""
ask_answer = g_raw.query(PREFIXES + ask_query).askAnswer
print("TRUY VẤN 10: DATASET CÓ LIÊN KẾT NGOÀI?", ask_answer)
with open(os.path.join(RESULTS_DIR, "query_10_ask_external_links.txt"), "w", encoding="utf-8") as f:
    f.write(f"ASK: dataset has owl:sameAs links to Wikidata/DBpedia = {ask_answer}\n")

TRUY VẤN 10: DATASET CÓ LIÊN KẾT NGOÀI? True


### Truy vấn 11: CONSTRUCT đồ thị ngân hàng–dịch vụ theo schema.org
CONSTRUCT tạo đồ thị mới. Ở đây chuyển quan hệ nội bộ sang từ vựng chuẩn `schema:makesOffer`.

In [17]:
construct_query = """
CONSTRUCT {
    ?bank a schema:BankOrCreditUnion ;
          schema:name ?name ;
          schema:makesOffer ?service .
    ?service a schema:Service ;
             schema:name ?serviceName .
}
WHERE {
    ?bank a :Bank ;
          :nameEn ?name ;
          :providesService ?service .
    ?service rdfs:label ?serviceName .
    FILTER(LANG(?serviceName) = "en")
    FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
}
"""
service_graph = g_raw.query(PREFIXES + construct_query).graph
service_graph.bind("schema", "https://schema.org/")
out = os.path.join(RESULTS_DIR, "query_11_bank_service_graph.ttl")
service_graph.serialize(out, format="turtle")
print("TRUY VẤN 11: số bộ ba được tạo:", len(service_graph))
print(service_graph.serialize(format="turtle")[:1200])

TRUY VẤN 11: số bộ ba được tạo: 280


@prefix schema: <https://schema.org/> .

<https://data.example.org/vietnam-banking/bank/abbank> a schema:BankOrCreditUnion ;
    schema:makesOffer <https://data.example.org/vietnam-banking/service/card>,
        <https://data.example.org/vietnam-banking/service/corporate-banking>,
        <https://data.example.org/vietnam-banking/service/deposit-savings>,
        <https://data.example.org/vietnam-banking/service/digital-banking>,
        <https://data.example.org/vietnam-banking/service/payment-transfer>,
        <https://data.example.org/vietnam-banking/service/retail-loan>,
        <https://data.example.org/vietnam-banking/service/trade-finance> ;
    schema:name "An Binh Commercial Joint Stock Bank"@en .

<https://data.example.org/vietnam-banking/bank/acb> a schema:BankOrCreditUnion ;
    schema:makesOffer <https://data.example.org/vietnam-banking/service/card>,
        <https://data.example.org/vietnam-banking/service/corporate-banking>,
        <https://data.example.org/vietnam-ba

### Truy vấn 12: Ngân hàng chưa có liên kết ngoài
Phủ định bằng FILTER NOT EXISTS. Dùng để báo cáo trung thực độ phủ liên kết.

In [18]:
df12 = run_query(
    "TRUY VẤN 12: NGÂN HÀNG CHƯA CÓ LIÊN KẾT NGOÀI",
    """
    SELECT ?id ?nameVi
    WHERE {
        ?bank a :Bank ;
              :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
        FILTER NOT EXISTS {
            ?bank owl:sameAs ?external .
            FILTER(STRSTARTS(STR(?external), "http://www.wikidata.org/entity/")
                || STRSTARTS(STR(?external), "http://dbpedia.org/resource/"))
        }
    }
    ORDER BY ?nameVi
    """,
    "query_12_banks_without_external_links.csv",
    graph=g_raw,
)

TRUY VẤN 12: NGÂN HÀNG CHƯA CÓ LIÊN KẾT NGOÀI
Số kết quả: 6


,id,nameVi
0,BACABANK,Ngân hàng TMCP Bắc Á
1,NCB,Ngân hàng TMCP Quốc Dân
2,VCBNEO,Ngân hàng TNHH MTV Ngoại thương Công nghệ số
3,SHINHANVN,Ngân hàng TNHH MTV Shinhan Việt Nam
4,MBV,Ngân hàng TNHH MTV Việt Nam Hiện Đại
5,GPBANK,Ngân hàng Thương mại TNHH MTV Kỷ Nguyên Thịnh Vượng


### Truy vấn 13: Ngân hàng nhà nước hoặc ngân hàng nước ngoài
UNION hai lớp được suy luận, kèm BIND để gắn nhãn nhóm.

In [19]:
df13 = run_query(
    "TRUY VẤN 13: NGÂN HÀNG NHÀ NƯỚC HOẶC NGÂN HÀNG NƯỚC NGOÀI",
    """
    SELECT ?id ?nameVi ?category
    WHERE {
        { ?bank a :StateOwnedBank . BIND("State-owned" AS ?category) }
        UNION
        { ?bank a :ForeignBank . BIND("Foreign-owned" AS ?category) }
        ?bank :bankId ?id ;
              :nameVi ?nameVi .
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    ORDER BY ?category ?nameVi
    """,
    "query_13_union_bank_categories.csv",
    graph=g,
)

TRUY VẤN 13: NGÂN HÀNG NHÀ NƯỚC HOẶC NGÂN HÀNG NƯỚC NGOÀI
Số kết quả: 6


,id,nameVi,category
0,SHINHANVN,Ngân hàng TNHH MTV Shinhan Việt Nam,Foreign-owned
1,AGRIBANK,Ngân hàng Nông nghiệp và Phát triển Nông thôn Việt Nam,State-owned
2,VIETINBANK,Ngân hàng TMCP Công thương Việt Nam,State-owned
3,VIETCOMBANK,Ngân hàng TMCP Ngoại thương Việt Nam,State-owned
4,BIDV,Ngân hàng TMCP Đầu tư và Phát triển Việt Nam,State-owned
5,VCBNEO,Ngân hàng TNHH MTV Ngoại thương Công nghệ số,State-owned


### Truy vấn 14: DESCRIBE ngân hàng ACB
DESCRIBE trả về toàn bộ mô tả của một tài nguyên. Chạy trên đồ thị gốc để kết quả gọn.

In [20]:
acb_graph = g_raw.query(PREFIXES + "DESCRIBE bank:acb").graph
out = os.path.join(RESULTS_DIR, "query_14_describe_acb.ttl")
acb_graph.serialize(out, format="turtle")
print("TRUY VẤN 14: số bộ ba mô tả ACB:", len(acb_graph))
print(acb_graph.serialize(format="turtle"))

TRUY VẤN 14: số bộ ba mô tả ACB: 27
@prefix : <https://data.example.org/vietnam-banking/> .
@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix schema: <https://schema.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://data.example.org/vietnam-banking/bank/acb> a owl:NamedIndividual,
        :Bank ;
    rdfs:label "Asia Commercial Joint Stock Bank"@en,
        "Ngân hàng TMCP Á Châu"@vi ;
    dcterms:source <https://en.wikipedia.org/wiki/List_of_banks_in_Vietnam> ;
    owl:sameAs <http://dbpedia.org/resource/Asia_Commercial_Bank>,
        <http://www.wikidata.org/entity/Q727019> ;
    :abbreviation "ACB" ;
    :bankId "ACB" ;
    :foundedYear 1993 ;
    :hasBankType <https://data.example.org/vietnam-banking/bank-type/joint-stock-commercial> ;
    :hasHeadquarters <https://data.example.org/vietnam-banking/location/ho-chi-minh-city> ;
    :nameEn "Asia Commercial J

### Truy vấn 15: Truy vấn liên kết (federated) tới Wikidata bằng SERVICE
Đây là truy vấn chứng minh liên kết 5★ dùng được: lấy nhãn và ngày thành lập (P571) từ Wikidata rồi đặt cạnh năm thành lập trong dữ liệu nội bộ. Cần kết nối Internet. Nếu không kết nối được, notebook vẫn chạy tiếp.

In [21]:
wikidata_iris = sorted({
    str(o) for s, o in g_raw.subject_objects(OWL.sameAs)
    if str(s).startswith("https://data.example.org/vietnam-banking/bank/")
    and str(o).startswith("http://www.wikidata.org/entity/")
})
values = " ".join(f"<{iri}>" for iri in wikidata_iris)
print("Số IRI Wikidata gửi đi:", len(wikidata_iris))

query_15 = f"""
SELECT ?id ?nameVi ?foundedYear ?wikidata ?wikidataLabel ?wikidataInception
WHERE {{
    ?bank a :Bank ;
          :bankId ?id ;
          :nameVi ?nameVi ;
          :foundedYear ?foundedYear ;
          owl:sameAs ?wikidata .
    FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    SERVICE <https://query.wikidata.org/sparql> {{
        VALUES ?wikidata {{ {values} }}
        ?wikidata rdfs:label ?wikidataLabel .
        FILTER(LANG(?wikidataLabel) = "en")
        OPTIONAL {{ ?wikidata wdt:P571 ?wikidataInception }}
    }}
}}
ORDER BY ?nameVi
"""
try:
    df15 = run_query("TRUY VẤN 15: BỔ SUNG DỮ LIỆU TỪ WIKIDATA", query_15,
                     "query_15_wikidata_service.csv", graph=g_raw)
except Exception as e:
    df15 = None
    print("Không truy cập được Wikidata từ môi trường hiện tại:", type(e).__name__)
    print("Hãy chạy lại ô này khi có Internet (ví dụ trên Google Colab).")

Số IRI Wikidata gửi đi: 24


Không truy cập được Wikidata từ môi trường hiện tại: URLError
Hãy chạy lại ô này khi có Internet (ví dụ trên Google Colab).


### Truy vấn 16: SPARQL terminal
Hàm `sparql()` cho phép gõ truy vấn bất kỳ. Ví dụ: số ngân hàng theo nơi đặt trụ sở, kèm liên kết DBpedia của địa điểm.

In [22]:
df16 = sparql("""
    SELECT ?city ?dbpedia (COUNT(DISTINCT ?bank) AS ?numberOfBanks)
    WHERE {
        ?bank a :Bank ;
              :hasHeadquarters ?loc .
        ?loc rdfs:label ?city ;
             owl:sameAs ?dbpedia .
        FILTER(LANG(?city) = "vi")
        FILTER(STRSTARTS(STR(?dbpedia), "http://dbpedia.org/resource/"))
        FILTER(STRSTARTS(STR(?bank), "https://data.example.org/vietnam-banking/bank/"))
    }
    GROUP BY ?city ?dbpedia
    ORDER BY DESC(?numberOfBanks)
""", g_raw)
df16.to_csv(os.path.join(RESULTS_DIR, "query_16_terminal_banks_by_city.csv"), index=False, encoding="utf-8-sig")
display(df16)

,city,dbpedia,numberOfBanks
0,Hà Nội,http://dbpedia.org/resource/Hanoi,20
1,Thành phố Hồ Chí Minh,http://dbpedia.org/resource/Ho_Chi_Minh_City,10


## 3. Đóng gói kết quả

In [23]:
result_files = sorted(f for f in os.listdir(RESULTS_DIR) if f.startswith("query_"))
zip_path = os.path.join(RESULTS_DIR, "vietnam_banking_sparql_results.zip")
with zipfile.ZipFile(zip_path, "w") as z:
    for f in result_files:
        z.write(os.path.join(RESULTS_DIR, f), f)
print("Đã đóng gói", len(result_files), "file vào", zip_path)
for f in result_files:
    print(" -", f)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

Đã đóng gói 15 file vào ../results/vietnam_banking_sparql_results.zip
 - query_01_all_banks.csv
 - query_02_state_owned_banks.csv
 - query_03_hanoi_headquarters.csv
 - query_04_digital_banks.csv
 - query_05_established_before_1991.csv
 - query_06_external_links.csv
 - query_07_inverse_service.csv
 - query_08_inverse_headquarters.csv
 - query_09_count_banks_by_type.csv
 - query_10_ask_external_links.txt
 - query_11_bank_service_graph.ttl
 - query_12_banks_without_external_links.csv
 - query_13_union_bank_categories.csv
 - query_14_describe_acb.ttl
 - query_16_terminal_banks_by_city.csv
